# Readiness Aggregator Notebook
Load feature-scoped datasets (inventory + gaps), compute readiness matrices, and export dashboard-friendly snippets.

In [ ]:
from __future__ import annotations

import json
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from statistics import mean
from typing import Iterable, Sequence

FEATURE_DIR = Path("/home/user/Code/udocket/specs/001-ai-refactor-plan")
DATA_DIR = FEATURE_DIR / "data" / "readiness"

def load_json(path: Path):
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)

inventory = load_json(DATA_DIR / "inventory.json")
gaps = load_json(DATA_DIR / "gaps.json")
len(inventory), len(gaps)


In [ ]:
# Build readiness matrix grouped by status
status_counts = Counter(item["status"] for item in inventory)
status_counts


In [ ]:
# Score averages for architecture/compliance/observability
def avg(key: str) -> float:
    values = [item[key] for item in inventory]
    return round(mean(values), 2)

avg_arch = avg("architecture_score")
avg_comp = avg("compliance_score")
avg_obs = avg("observability_score")
avg_arch, avg_comp, avg_obs


In [ ]:
# Export dashboard snippet
export_payload = {
    "generated_at": __import__("datetime").datetime.utcnow().isoformat() + "Z",
    "stage_status_counts": status_counts,
    "score_averages": {
        "architecture": avg_arch,
        "compliance": avg_comp,
        "observability": avg_obs,
    },
    "active_gaps": [gap for gap in gaps if gap["status"] != "closed"],
}

EXPORT_PATH = FEATURE_DIR / "reports" / "readiness_dashboard_snapshot.json"
EXPORT_PATH.write_text(json.dumps(export_payload, indent=2) + "\n", encoding="utf-8")
EXPORT_PATH
